In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm

import os, json, random, copy, time, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from concurrent.futures import ThreadPoolExecutor
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- dataset -> local disk (parallel, resumable) ----
LOCAL_DATA = "/tmp/tea_leaf_local"
N_WORKERS  = 32

def _count_files(p):
    return sum(len(f) for _, _, f in os.walk(p)) if os.path.isdir(p) else 0

n_src = _count_files(TEA_LEAF_ROOT)
n_dst = _count_files(LOCAL_DATA)

if n_dst < n_src:
    t0 = time.time()
    tasks = []
    for root, _, files_ in os.walk(TEA_LEAF_ROOT):
        rel = os.path.relpath(root, TEA_LEAF_ROOT)
        dst_dir = LOCAL_DATA if rel == '.' else os.path.join(LOCAL_DATA, rel)
        os.makedirs(dst_dir, exist_ok=True)
        for f in files_:
            src, dst = os.path.join(root, f), os.path.join(dst_dir, f)
            if not os.path.exists(dst):
                tasks.append((src, dst))
    print(f"Copying {len(tasks)} files ({n_dst}/{n_src} local) with {N_WORKERS} workers...")
    def _cp(t):
        try:
            shutil.copyfile(t[0], t[1]); return True
        except Exception:
            return False
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        ok = list(tqdm(ex.map(_cp, tasks), total=len(tasks), desc="copy"))
    print(f" Copied {sum(ok)}/{len(tasks)} in {time.time()-t0:.0f}s")
else:
    print(f" Local cache complete ({n_dst} files)")

classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES     = len(classes)
CLASS_TO_IDX    = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
print(f"Classes ({NUM_CLASSES}): {classes}")

dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn

def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads,
                                       C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = ((cls_q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = ((q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x

dv1_last_attn.forward = _dv1_patched_forward

tf_dv1 = T.Compose([T.Resize((224,224)), T.ToTensor(),
                    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_train = T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.RandomRotation(20), T.ColorJitter(0.2,0.2,0.2), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_test = T.Compose([T.Resize((224,224)), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

DV1_GRID = 14
N_HEADS  = 6
topk     = max(1, int(0.1 * DV1_GRID * DV1_GRID))

with torch.no_grad():
    _ = dino_v1(torch.randn(1,3,224,224).to(device))
print(f"Attention hook OK: {_dv1_store['cls_attn'].shape}")

all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg','.jpeg','.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname),
                'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1,
            })

class SimpleDS(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs = imgs; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        return self.transform(Image.open(self.imgs[idx]['path']).convert('RGB')), self.labels[idx]

class TestDS(Dataset):
    def __init__(self, imgs, transform):
        self.imgs = imgs; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        return (self.transform(Image.open(self.imgs[idx]['path']).convert('RGB')),
                self.imgs[idx]['is_diseased'])

n_dis = sum(x['is_diseased'] for x in all_images)
print(f"\n Ready: {len(all_images)} images | Diseased: {n_dis}, Healthy: {len(all_images)-n_dis}")

In [ ]:
import os, shutil, sys
from google.colab import files

if not os.path.exists("/content/multiclass_cluster.py"):
    print("Select multiclass_cluster.py:")
    up = files.upload()
    src = list(up.keys())[0]
    if os.path.abspath(src) != "/content/multiclass_cluster.py":
        shutil.move(src, "/content/multiclass_cluster.py")

sys.modules.pop("multiclass_cluster", None)
import multiclass_cluster as mc
print("OK:", mc.__file__)

In [ ]:
import numpy as np, torch, json, os
from PIL import Image
from tqdm import tqdm
from sklearn.cluster import KMeans
import multiclass_cluster as mc

# --- frozen DINO [CLS] features, NO labels used ---
feats, labels = [], []
with torch.no_grad():
    for rec in tqdm(all_images, desc="features"):
        try:
            im = Image.open(rec['path']).convert('RGB')
            f = dino_v1(tf_dv1(im).unsqueeze(0).to(device)).cpu().numpy()[0]
            feats.append(f); labels.append(rec['class_idx'])
        except Exception:
            continue

X = np.array(feats, dtype=np.float32)
y = np.array(labels)
X /= (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
print(f"\nFeatures: {X.shape}")

np.save(os.path.join(RESULTS_DIR, 'dino_features.npy'), X)
np.save(os.path.join(RESULTS_DIR, 'dino_labels.npy'), y)

# --- cluster into 7 groups ---
K = len(classes)
km = KMeans(n_clusters=K, n_init=20, random_state=0).fit(X)

res = mc.report(y, km.labels_, X, km.cluster_centers_, classes, K)

with open(os.path.join(RESULTS_DIR, 'multiclass_clustering.json'), 'w') as f:
    json.dump(res, f, indent=2)
print("Saved multiclass_clustering.json")

In [ ]:
import numpy as np, json, os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
import multiclass_cluster as mc

X = np.load(os.path.join(RESULTS_DIR, 'dino_features.npy'))
y = np.load(os.path.join(RESULTS_DIR, 'dino_labels.npy'))
K = len(classes)

# 1. supervised linear probe = UPPER BOUND (uses all labels)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
probe = LogisticRegression(max_iter=2000, n_jobs=-1).fit(Xtr, ytr)
probe_acc = probe.score(Xte, yte)

# 2. clustering stability across seeds
hung, oned = [], []
for s in [0, 1, 2]:
    km = KMeans(n_clusters=K, n_init=20, random_state=s).fit(X)
    h, _ = mc.hungarian_accuracy(y, km.labels_, K)
    o, _, _ = mc.oneshot_naming_accuracy(y, km.labels_, X, km.cluster_centers_)
    hung.append(h); oned.append(o)

print("=" * 62)
print("MULTI-CLASS REFERENCE POINTS")
print("=" * 62)
print(f"  chance (1/{K})                     : {100/K:5.1f}%")
print(f"  unsupervised clustering (Hungarian): {np.mean(hung)*100:5.1f}% ± {np.std(hung)*100:.1f}")
print(f"  one-shot naming ({K} labels)        : {np.mean(oned)*100:5.1f}% ± {np.std(oned)*100:.1f}")
print(f"  supervised linear probe (all labels): {probe_acc*100:5.1f}%   [upper bound]")
print("=" * 62)
print(f"  -> clustering recovers "
      f"{np.mean(hung)/probe_acc*100:.0f}% of supervised performance with ZERO labels")

with open(os.path.join(RESULTS_DIR, 'multiclass_reference.json'), 'w') as f:
    json.dump({'chance': 1/K,
               'hungarian_mean': float(np.mean(hung)), 'hungarian_std': float(np.std(hung)),
               'oneshot_mean': float(np.mean(oned)), 'oneshot_std': float(np.std(oned)),
               'supervised_probe': float(probe_acc)}, f, indent=2)
print("Saved multiclass_reference.json")

In [ ]:
import numpy as np, os
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.decomposition import PCA
import multiclass_cluster as mc

X = np.load(os.path.join(RESULTS_DIR, 'dino_features.npy'))
y = np.load(os.path.join(RESULTS_DIR, 'dino_labels.npy'))
K = len(classes)

def evaluate(name, pred, cents):
    h, _ = mc.hungarian_accuracy(y, pred, K)
    o, _, n = mc.oneshot_naming_accuracy(y, pred, X, cents)
    print(f"{name:<34s} Hungarian {h*100:5.1f}%  one-shot {o*100:5.1f}%  named {n}/{K}")
    return h

# baseline
km = KMeans(K, n_init=20, random_state=0).fit(X)
evaluate("k-means (baseline)", km.labels_, km.cluster_centers_)

# PCA whitening — usually the single biggest win on DINO features
for d in [32, 64, 128]:
    Z = PCA(n_components=d, whiten=True, random_state=0).fit_transform(X)
    k2 = KMeans(K, n_init=20, random_state=0).fit(Z)
    cents = np.stack([X[k2.labels_ == c].mean(0) for c in range(K)])
    evaluate(f"PCA-{d} whiten + k-means", k2.labels_, cents)

# spherical k-means (cosine geometry — matches L2-normalised features)
ac = AgglomerativeClustering(n_clusters=K, metric='cosine', linkage='average').fit(X)
cents = np.stack([X[ac.labels_ == c].mean(0) for c in range(K)])
evaluate("agglomerative (cosine)", ac.labels_, cents)

# over-cluster then merge: gives small classes their own cluster
for k_over in [10, 14, 20]:
    ko = KMeans(k_over, n_init=20, random_state=0).fit(X)
    # merge the over-clusters down to K by their centroid similarity
    merge = AgglomerativeClustering(n_clusters=K, metric='cosine',
                                    linkage='average').fit(ko.cluster_centers_)
    pred = merge.labels_[ko.labels_]
    cents = np.stack([X[pred == c].mean(0) for c in range(K)])
    evaluate(f"over-cluster k={k_over} -> merge to {K}", pred, cents)

In [ ]:
import numpy as np, os, json
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import multiclass_cluster as mc

X = np.load(os.path.join(RESULTS_DIR, 'dino_features.npy'))
y = np.load(os.path.join(RESULTS_DIR, 'dino_labels.npy'))
K = len(classes)

hung, ones, named = [], [], []
for s in [0, 1, 2, 3, 4]:
    Z  = PCA(n_components=128, whiten=True, random_state=s).fit_transform(X)
    km = KMeans(K, n_init=20, random_state=s).fit(Z)
    cents = np.stack([X[km.labels_ == c].mean(0) for c in range(K)])
    h, _    = mc.hungarian_accuracy(y, km.labels_, K)
    o, _, n = mc.oneshot_naming_accuracy(y, km.labels_, X, cents)
    hung.append(h); ones.append(o); named.append(n)
    print(f"seed {s}: Hungarian {h*100:.1f}%  one-shot {o*100:.1f}%  named {n}/{K}")

print(f"\nPCA-128 + k-means over 5 seeds:")
print(f"  Hungarian : {np.mean(hung)*100:.1f}% ± {np.std(hung)*100:.1f}")
print(f"  one-shot  : {np.mean(ones)*100:.1f}% ± {np.std(ones)*100:.1f}")
print(f"  classes named: {np.mean(named):.1f}/{K}")

# final cluster composition for the paper
Z  = PCA(n_components=128, whiten=True, random_state=0).fit_transform(X)
km = KMeans(K, n_init=20, random_state=0).fit(Z)
cents = np.stack([X[km.labels_ == c].mean(0) for c in range(K)])
res = mc.report(y, km.labels_, X, cents, classes, K)
res.update({'method': 'PCA-128 whiten + kmeans',
            'hungarian_std': float(np.std(hung)), 'oneshot_std': float(np.std(ones))})
with open(os.path.join(RESULTS_DIR, 'multiclass_clustering.json'), 'w') as f:
    json.dump(res, f, indent=2)
print("Saved")

In [ ]:
import numpy as np, os, json
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import multiclass_cluster as mc

X = np.load(os.path.join(RESULTS_DIR, 'dino_features.npy'))
y = np.load(os.path.join(RESULTS_DIR, 'dino_labels.npy'))
K = len(classes)

def run(Z, km_seed):
    km = KMeans(K, n_init=20, random_state=km_seed).fit(Z)
    cents = np.stack([X[km.labels_ == c].mean(0) for c in range(K)])
    h, _    = mc.hungarian_accuracy(y, km.labels_, K)
    o, _, n = mc.oneshot_naming_accuracy(y, km.labels_, X, cents)
    return h, o, n, km.labels_, cents

# ---- A. PCA FIXED (seed 0), vary only k-means ----
print("A) PCA fixed at random_state=0, varying k-means seed")
Z_fixed = PCA(n_components=128, whiten=True, random_state=0).fit_transform(X)
A = []
for s in range(5):
    h, o, n, _, _ = run(Z_fixed, s)
    A.append((h, o, n))
    print(f"   km seed {s}: Hungarian {h*100:5.1f}%  one-shot {o*100:5.1f}%  named {n}/{K}")
ha, oa, na = zip(*A)
print(f"   -> Hungarian {np.mean(ha)*100:.1f}% ± {np.std(ha)*100:.1f} | "
      f"one-shot {np.mean(oa)*100:.1f}% ± {np.std(oa)*100:.1f} | named {np.mean(na):.1f}/{K}")

# ---- B. PCA varied, k-means fixed (isolates PCA's contribution) ----
print("\nB) k-means fixed at random_state=0, varying PCA seed")
B = []
for s in range(5):
    Z = PCA(n_components=128, whiten=True, random_state=s).fit_transform(X)
    h, o, n, _, _ = run(Z, 0)
    B.append((h, o, n))
    print(f"   pca seed {s}: Hungarian {h*100:5.1f}%  one-shot {o*100:5.1f}%  named {n}/{K}")
hb, ob, nb = zip(*B)
print(f"   -> Hungarian {np.mean(hb)*100:.1f}% ± {np.std(hb)*100:.1f} | "
      f"one-shot {np.mean(ob)*100:.1f}% ± {np.std(ob)*100:.1f} | named {np.mean(nb):.1f}/{K}")

# ---- C. deterministic PCA (full SVD, no randomness at all) ----
print("\nC) deterministic PCA (svd_solver='full'), varying only k-means")
Z_det = PCA(n_components=128, whiten=True, svd_solver='full').fit_transform(X)
C = []
for s in range(5):
    h, o, n, _, _ = run(Z_det, s)
    C.append((h, o, n))
    print(f"   km seed {s}: Hungarian {h*100:5.1f}%  one-shot {o*100:5.1f}%  named {n}/{K}")
hc, oc, nc = zip(*C)
print(f"   -> Hungarian {np.mean(hc)*100:.1f}% ± {np.std(hc)*100:.1f} | "
      f"one-shot {np.mean(oc)*100:.1f}% ± {np.std(oc)*100:.1f} | named {np.mean(nc):.1f}/{K}")

print("\n" + "="*66)
print("Where does the variance live?")
print(f"  vary k-means only (PCA fixed)   : one-shot std {np.std(oa)*100:.1f}")
print(f"  vary PCA only (k-means fixed)   : one-shot std {np.std(ob)*100:.1f}")
print(f"  deterministic PCA, vary k-means : one-shot std {np.std(oc)*100:.1f}")
print("="*66)
print("If (C) is tight, we can legitimately fix PCA (it IS deterministic on a")
print("fixed dataset up to sign) and report the k-means-only variance.")

In [ ]:
import numpy as np, os, json
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import multiclass_cluster as mc

X = np.load(os.path.join(RESULTS_DIR, 'dino_features.npy'))
y = np.load(os.path.join(RESULTS_DIR, 'dino_labels.npy'))
K = len(classes)
Z = PCA(n_components=128, whiten=True, svd_solver='full').fit_transform(X)

def score(labels):
    cents = np.stack([X[labels == c].mean(0) for c in range(K)])
    h, _    = mc.hungarian_accuracy(y, labels, K)
    o, _, n = mc.oneshot_naming_accuracy(y, labels, X, cents)
    return h, o, n

# --- 1. many more restarts (the cheap, principled fix) ---
for n_init in [50, 200, 500]:
    r = [score(KMeans(K, n_init=n_init, init='k-means++', random_state=s).fit(Z).labels_)
         for s in range(5)]
    h, o, n = zip(*r)
    print(f"n_init={n_init:<4d} Hungarian {np.mean(h)*100:5.1f}% ± {np.std(h)*100:4.1f} | "
          f"one-shot {np.mean(o)*100:5.1f}% ± {np.std(o)*100:4.1f} | named {np.mean(n):.1f}/{K}")

# --- 2. select the run by INERTIA (label-free!) — this is the deployable rule ---
print("\nInertia-selected (no labels used to choose the run):")
best, best_in = None, np.inf
for s in range(20):
    km = KMeans(K, n_init=50, init='k-means++', random_state=s).fit(Z)
    if km.inertia_ < best_in:
        best_in, best = km.inertia_, km.labels_
h, o, n = score(best)
print(f"  Hungarian {h*100:.1f}%  one-shot {o*100:.1f}%  named {n}/{K}  (inertia={best_in:.1f})")